In [0]:
from pyspark.sql import functions as F

df_bronze = spark.table("bronze_voos_anac")

cols = df_bronze.columns
col_empresa = [c for c in cols if "Empresa" in c][0]
col_voo = [c for c in cols if "Voo" in c and "Situa" not in c][0]
col_linha = [c for c in cols if "Linha" in c][0]
col_origem = [c for c in cols if "Origem" in c][0]
col_destino = [c for c in cols if "Destino" in c][0]
col_situacao = [c for c in cols if "Situa" in c][0]
col_justificativa = [c for c in cols if "Justificativa" in c][0]

# Tabelas de referência (Dimensões) para enriquecimento na camada Silver
dim_empresas = spark.createDataFrame([
    ("TAM", "LATAM Brasil (TAM)"),
    ("AZU", "Azul Linhas Aéreas (AZU)"),
    ("GLO", "GOL Linhas Aéreas (GLO)"),
    ("LAN", "LATAM Airlines Chile (LAN)"),
    ("ARG", "Aerolíneas Argentinas (ARG)"),
    ("ACN", "Azul Conecta (ACN)"),
    ("TAP", "TAP Air Portugal (TAP)"),
    ("CMP", "Copa Airlines (CMP)"),
    ("JES", "JetSMART Argentina (JES)"),
    ("FBZ", "Flybondi (FBZ)"),
    ("SKU", "Sky Airline (SKU)"),
    ("AAL", "American Airlines (AAL)"),
    ("LTG", "LATAM Cargo Brasil (LTG)"),
    ("JAT", "JetSMART Chile (JAT)"),
    ("AVA", "Avianca (AVA)"),
    ("IBE", "Iberia (IBE)"),
    ("LPE", "LATAM Airlines Perú (LPE)")
], ["cod_empresa", "nome_empresa_traduzido"])

dim_aeroportos = spark.createDataFrame([
    ("KMIA", "Miami (KMIA)"),
    ("SBKP", "Viracopos/Campinas (SBKP)"),
    ("SBGR", "Guarulhos (SBGR)"),
    ("SBFI", "Foz do Iguaçu (SBFI)"),
    ("SAEZ", "Buenos Aires/Ezeiza (SAEZ)"),
    ("SBFL", "Florianópolis (SBFL)"),
    ("SABE", "Buenos Aires/Aeroparque (SABE)"),
    ("KJFK", "Nova York/JFK (KJFK)"),
    ("SBVT", "Vitória (SBVT)"),
    ("SACO", "Córdoba (SACO)"),
    ("SBGL", "Rio de Janeiro/Galeão (SBGL)")
], ["cod_aeroporto", "nome_aeroporto_traduzido"])

df_base = (df_bronze
    .select(
        F.col(col_empresa).alias("empresa_aerea"),
        F.col(col_voo).alias("numero_voo"),
        F.col(col_linha).alias("tipo_linha"),
        F.col(col_origem).alias("aeroporto_origem"),
        F.col(col_destino).alias("aeroporto_destino"),
        F.try_to_timestamp("Partida_Prevista").alias("partida_prevista"),
        F.try_to_timestamp("Partida_Real").alias("partida_real"),
        F.try_to_timestamp("Chegada_Prevista").alias("chegada_prevista"),
        F.try_to_timestamp("Chegada_Real").alias("chegada_real"),
        F.col(col_situacao).alias("situacao_voo"),
        F.col(col_justificativa).alias("codigo_justificativa")
    )
    .filter(F.col("empresa_aerea").isNotNull())
    .withColumn("atraso_chegada_min", 
        F.round((F.unix_timestamp("chegada_real") - F.unix_timestamp("chegada_prevista")) / 60, 2)
    )
    .withColumn("status_pontualidade",
        F.when(F.col("situacao_voo") == "CANCELADO", "Cancelado")
         .when(F.col("atraso_chegada_min") > 15, "Atrasado (>15 min)")
         .otherwise("No Horário")
    )
    .dropDuplicates()
)

# Enriquece a Silver com os nomes completos usando Broadcast Join
df_silver = (df_base
    .join(F.broadcast(dim_empresas), df_base.empresa_aerea == dim_empresas.cod_empresa, "left")
    .join(F.broadcast(dim_aeroportos.alias("orig")), df_base.aeroporto_origem == F.col("orig.cod_aeroporto"), "left")
    .join(F.broadcast(dim_aeroportos.alias("dest")), df_base.aeroporto_destino == F.col("dest.cod_aeroporto"), "left")
    .withColumn("nome_companhia", F.coalesce(F.col("nome_empresa_traduzido"), F.col("empresa_aerea")))
    .withColumn("origem_descritiva", F.coalesce(F.col("orig.nome_aeroporto_traduzido"), F.col("aeroporto_origem")))
    .withColumn("destino_descritiva", F.coalesce(F.col("dest.nome_aeroporto_traduzido"), F.col("aeroporto_destino")))
    .withColumn("rota_descritiva", F.concat(F.col("origem_descritiva"), F.lit(" -> "), F.col("destino_descritiva")))
    .drop("cod_empresa", "nome_empresa_traduzido", "cod_aeroporto", "nome_aeroporto_traduzido")
)

df_silver.write.format("delta").option("overwriteSchema", "true").mode("overwrite").saveAsTable("silver_voos_anac")

print(f"Total de voos tratados na camada Silver: {df_silver.count()}")
display(df_silver.select("nome_companhia", "rota_descritiva", "situacao_voo", "atraso_chegada_min", "status_pontualidade").limit(10))

Total de voos tratados na camada Silver: 258301


nome_companhia,rota_descritiva,situacao_voo,atraso_chegada_min,status_pontualidade
GOL Linhas Aéreas (GLO),Rio de Janeiro/Galeão (SBGL) -> SBSV,REALIZADO,1.0,No Horário
GOL Linhas Aéreas (GLO),Rio de Janeiro/Galeão (SBGL) -> SBSV,REALIZADO,8.0,No Horário
GOL Linhas Aéreas (GLO),Rio de Janeiro/Galeão (SBGL) -> SBSV,REALIZADO,2.0,No Horário
GOL Linhas Aéreas (GLO),Foz do Iguaçu (SBFI) -> Rio de Janeiro/Galeão (SBGL),REALIZADO,-26.0,No Horário
GOL Linhas Aéreas (GLO),Foz do Iguaçu (SBFI) -> Rio de Janeiro/Galeão (SBGL),REALIZADO,-13.0,No Horário
GOL Linhas Aéreas (GLO),Foz do Iguaçu (SBFI) -> Rio de Janeiro/Galeão (SBGL),REALIZADO,-26.0,No Horário
GOL Linhas Aéreas (GLO),Foz do Iguaçu (SBFI) -> Rio de Janeiro/Galeão (SBGL),REALIZADO,2.0,No Horário
GOL Linhas Aéreas (GLO),Florianópolis (SBFL) -> Rio de Janeiro/Galeão (SBGL),REALIZADO,-12.0,No Horário
GOL Linhas Aéreas (GLO),Rio de Janeiro/Galeão (SBGL) -> SBAR,REALIZADO,-16.0,No Horário
GOL Linhas Aéreas (GLO),Rio de Janeiro/Galeão (SBGL) -> SBAR,REALIZADO,-23.0,No Horário
